In [ ]:
from hyper_evolution import time_evolve, animate
from scipy.special import gamma
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', context='notebook')
sns.set_palette("Dark2")

In [ ]:
lam = 3
# x0, p0 = 0., np.sqrt(lam-0.25)
x0, p0 = 1, 0
breadth = 1000

In [ ]:
# domain = np.linspace(-breadth, breadth, 10000)
# dx = domain[1] - domain[0]
dx = .1
domain = np.arange(-breadth, breadth + dx, dx)

def sech2(x):
    ax = np.abs(x)
    e = np.exp(-ax)
    return (2 * e / (1 + e*e))**2

Vx = -lam * (lam + 1) * sech2(domain) + lam**2

c = np.sqrt(gamma(lam+0.5)/(np.sqrt(np.pi) * gamma(lam)))

y = domain - x0
ay = np.abs(y)
log_cosh = ay + np.log1p(np.exp(-2*ay)) - np.log(2)

psi_0 = np.exp(np.log(c) - lam * log_cosh)
psi_0 = psi_0.astype(np.complex128)
psi_0 *= np.exp(1j * p0 * domain)


In [ ]:
max_t = 10
dt = dx ** 2
T = int(max_t / dt) + 1
results = time_evolve(domain, psi_0, Vx, max_t, it=T, hyperbolic=False, lam_val=lam)

In [ ]:
t_eval = np.array(results['t'])
E_q = np.array(results['T']) + np.array(results['V'])
unbound_prob = np.abs(results['unbound_psi']) ** 2
unbound_x = np.array(results['unbound_x'])
bound_psi = [psi_avg-unbound_psi for psi_avg, unbound_psi in zip(results['psi_avg'], results['unbound_psi'])]
bound_x = [np.trapz(domain * np.abs(psi)**2, dx=dx) for psi in bound_psi]
left_diff = (unbound_prob[1:, 0] - unbound_prob[:-1, 0])
right_diff = (unbound_prob[1:, -1] - unbound_prob[:-1, -1])

In [ ]:
plt.plot(t_eval, results['err_psi'], label='err_psi')
plt.plot(t_eval, results['err_norm'], label='err_norm')
plt.legend()
plt.show()

In [ ]:
total_unbound = [np.trapz(abs(results['unbound_psi'][i])**2, dx=dx) for i in range(len(t_eval))]
plt.plot(t_eval, total_unbound)

In [ ]:
results = {'lam': [], 'p0': [], 'dif': []}
lams = [*range(2, 14, 2)]
l_p = [(l, l**2 + dp) for l in lams for dp in np.arange(0, 29 * (2 * l - 1), 2 * l - 1)]

In [ ]:
max_t = 100
t_eval = np.arange(0, max_t, 0.1)


In [ ]:
from scipy.integrate import solve_ivp

for lam, p0 in l_p:
    def hamilton_eqs(_, y):
        x, p = y
        dxdt = 2 * p
        dpdt = -2 * lam**2 * np.tanh(x) * (1 - np.tanh(x)**2) 
        return [dxdt, dpdt]

    sol = solve_ivp(
        hamilton_eqs,
        t_span=(0, max_t),
        y0=[x0, p0],
        t_eval=t_eval,
        method="DOP853",
        rtol=1e-10,
        atol=1e-12
    )

    x_classical = sol.y[0]
    p_classical = sol.y[1]
    results['lam'].append(lam)
    results['p0'].append((p0 - lam**2))
    results['dif'].append(p_classical[-1] - p_classical[0])

In [ ]:
import pandas as pd
results = pd.DataFrame(results)

In [ ]:
plt.scatter(results.p0, results.dif, c=results.lam)
plt.xlabel('Displacement $p_0 - \lambda^2$ from Unbound Energy ($\lambda^2$)')
plt.ylabel('Momentum Change')
plt.title("Classical $\Delta p$ for Free Particles (t=0 to t=$\infty$)")
plt.show()

In [ ]:
df = pd.DataFrame(fits)

In [ ]:
x = df.lam
y = df.a
plt.scatter(x, y)
np.polyfit(x, y, 1)

In [ ]:
from scipy.optimize import curve_fit
def model(x, a):
    return - a / (2 * (x + a))

fits = {'lam': [], 'a': [], 'b': []}
lambdas = sorted(results["lam"].unique())

cmap = plt.cm.viridis
colors = cmap(np.linspace(0, 1, len(lambdas)))

plt.figure(figsize=(8, 6))

for lam, color in zip(lambdas, colors):
    subset = results[results["lam"] == lam]
    
    x_data = subset["p0"].values
    y_data = subset["dif"].values
    
    if len(x_data) < 2:
        print(f"Not enough data points for lambda {lam}")
        continue
    popt, pcov = curve_fit(model, x_data, y_data)
    fits['lam'].append(lam)
    fits['a'].append(popt[0])
    # print(f"Lambda {lam} fitted parameters:", popt)
    plt.scatter(x_data, y_data, color=color, label=f"λ = {lam}, A={popt[0]:.3f}")
    x_fit = np.linspace(min(x_data), max(x_data), 200)
    y_fit = model(x_fit, *popt)
    
    plt.plot(x_fit, y_fit, color=color)

plt.xlabel('Displacement $p_0 - \lambda^2$ from Unbound Energy ($\lambda^2$)')
plt.ylabel('Measured Impulse')
plt.title(r'Fit of $f(x = p_0 - \lambda^2) = -\frac{A}{2(x + A)}$ for each $\lambda$')
plt.legend()
plt.show()

In [ ]:
plt.plot(t_eval, p_classical, color='g', label='Classical p')
plt.plot(t_eval, results['p'], color='b', label='Quantum ⟨p⟩')
plt.plot(t_eval, results['unbound_p'], color='r', linestyle='--', label='Continuum ⟨p⟩')

plt.title(f"Quantum Momentum vs. Classical Momentum")
plt.xlabel('Time')
plt.ylabel("Momentum")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6), dpi=100)
plt.plot(t_eval, x_classical, '--', label='Classical ⟨x⟩')
plt.plot(t_eval, results['x'], label='Quantum')
# plt.plot(t_eval, unbound_x, label='Continuum ⟨x⟩')
# plt.plot(t_eval, bound_x, label='Bound ⟨x⟩')

plt.title(f'Hyper Correspondence with Matched Energies ($x_0\\approx$ {x0:.2f}, $p_0 \\approx$ {p0:.2f}, $\\lambda = {lam}$)')
plt.ylabel("Position ⟨x⟩")
plt.xlabel("Time")
plt.grid(True, alpha=0.5)
plt.legend()
plt.show()

In [ ]:
skip = 100
animate(domain, (np.abs(results['psi_avg'])**2)[::skip], max_t, max_x=100, filename='psi_2.gif')